# B2-022-probabilistic-latent-models — Practice p20 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** variational-autoencoder

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

COMPONENT_MEANS = torch.tensor([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.5]])
N_PRIOR_SAMPLES = 512
INTERPOLATION_STEPS = 9
ATOL = 1e-6
RTOL = 1e-6

## Solution

Same `TinyVAE` and `negative_elbo` as p19 with `in_dim=2`, trained for exactly 800 updates on the 48 training rows (training rows guarded by hash during training).
After training, prior samples are decoded from $z\sim N(0,I)$ drawn with a separate seeded generator, and two held-out rows are used **only** as interpolation endpoints after training.

In [ ]:
import copy

DATASET = "mixture2d"


def train_rows():
    return latent_data.train_tensor(DATASET)


def heldout_rows():
    return latent_data.heldout_tensor(DATASET)


def split_hash_sets(name):
    hashes = latent_data.ROW_SHA256[name]
    train_part = {hashes[i] for i in latent_data.TRAIN_IDS[name]}
    heldout_part = {hashes[i] for i in latent_data.HELDOUT_IDS[name]}
    return train_part, heldout_part


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)


def install_training_row_guard(model):
    """Forward pre-hook: every row a training forward receives must hash into the training part."""
    seen = {"calls": 0, "rows": 0}

    def hook(module, args):
        batch = args[0]
        for row in batch:
            row_hash = latent_data.row_sha256(row)
            if row_hash not in TRAIN_HASHES or row_hash in HELDOUT_HASHES:
                raise AssertionError("a non-training row reached a training forward pass")
        seen["calls"] += 1
        seen["rows"] += batch.shape[0]

    return model.register_forward_pre_hook(hook), seen

def _check_inputs(x, x_hat, mu, logvar):
    for name, t in (("x", x), ("x_hat", x_hat), ("mu", mu), ("logvar", logvar)):
        if t.dim() != 2:
            raise ValueError(f"{name} must be 2-D")
    if x.shape != x_hat.shape:
        raise ValueError("x and x_hat shapes differ")
    if mu.shape != logvar.shape:
        raise ValueError("mu and logvar shapes differ")
    if x.shape[0] != mu.shape[0]:
        raise ValueError("batch sizes differ")


def negative_elbo_terms(x, x_hat, mu, logvar):
    _check_inputs(x, x_hat, mu, logvar)
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)                       # sum over features
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(dim=1)       # sum over latents
    return recon, kl


def negative_elbo(x, x_hat, mu, logvar):
    recon, kl = negative_elbo_terms(x, x_hat, mu, logvar)
    return (recon + kl).mean()                                         # mean over the batch

class TinyVAE(nn.Module):
    def __init__(self, in_dim=8, hidden=16, latent=2):
        super().__init__()
        self.hidden = nn.Linear(in_dim, hidden)
        self.mu_head = nn.Linear(hidden, latent)
        self.logvar_head = nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, in_dim))

    def encode(self, x):
        h = F.relu(self.hidden(x))
        return self.mu_head(h), self.logvar_head(h)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, eps):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * eps
        return self.decode(z), mu, logvar


def train_vae_for_sampling(model, batch, optimizer):
    eps_generator = torch.Generator().manual_seed(SEED)
    trace = []
    for _ in range(800):
        eps = torch.randn(batch.shape[0], 2, generator=eps_generator)
        optimizer.zero_grad(set_to_none=True)
        x_hat, mu, logvar = model(batch, eps)
        loss = negative_elbo(batch, x_hat, mu, logvar)
        trace.append(loss.item())
        loss.backward()
        optimizer.step()
    return trace


def sample_prior(model, n, generator):
    with torch.no_grad():
        return model.decode(torch.randn(n, 2, generator=generator))


def interpolate_latents(model, x_a, x_b, steps):
    with torch.no_grad():
        mu_a, _ = model.encode(x_a)
        mu_b, _ = model.encode(x_b)
        t = torch.linspace(0, 1, steps).unsqueeze(1)
        z_path = (1 - t) * mu_a + t * mu_b
        return z_path, model.decode(z_path)


torch.manual_seed(SEED)
model = TinyVAE(in_dim=2, hidden=16, latent=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
params_before = copy.deepcopy({name: p.detach() for name, p in model.named_parameters()})
guard, guard_stats = install_training_row_guard(model)
trace = train_vae_for_sampling(model, train_rows(), optimizer)
guard.remove()
changed = [name for name, p in model.named_parameters() if not torch.equal(p.detach(), params_before[name])]
last20 = sum(trace[-20:]) / 20

# Sampling
T = train_rows()
samples = sample_prior(model, N_PRIOR_SAMPLES, torch.Generator().manual_seed(SEED + 2))
assignment = torch.cdist(samples, COMPONENT_MEANS).argmin(dim=1)
counts = torch.bincount(assignment, minlength=3)

# Interpolation (held-out rows used only here, after training)
x_a = heldout_rows()[0:1]
x_b = heldout_rows()[1:2]
z_path, decoded_path = interpolate_latents(model, x_a, x_b, INTERPOLATION_STEPS)
with torch.no_grad():
    mu_a, _ = model.encode(x_a)
    mu_b, _ = model.encode(x_b)
    decoded_a = model.decode(mu_a)
    decoded_b = model.decode(mu_b)
steps_diff = z_path[1:] - z_path[:-1]
print(f"first total={trace[0]:.4f} last-20 mean={last20:.4f}")
print("sample mean", samples.mean(dim=0), "train mean", T.mean(dim=0))
print("sample std", samples.std(dim=0), "train std", T.std(dim=0))
print("component counts", counts.tolist())

### Tolerances versus exact probes

Probes 2–4 summarize 512 random prior draws pushed through an imperfect generative model, so their values carry both Monte Carlo noise and genuine model error and can only be required to lie within tolerances of the training statistics; the interpolation probes are deterministic algebra ($t=0$ and $t=1$ reproduce the endpoints, and a `linspace` path has equal steps), so they hold to floating-point precision.
A collapsed model whose decoder ignores $z$ would output nearly the same point for every prior sample, so it could pass the mean probe 2 but would fail the spread probe 3 and the coverage probe 4 (all samples landing near one component, or between components).
For the same collapse, the interpolation probes would still pass formally, but the encoder means `mu_a` and `mu_b` would nearly coincide and the decoded path would be essentially constant, which is why the sampling probes, not the interpolation probes, are the evidence that the latent is used.

### Answer check

In [ ]:
# Training certificate
assert train_rows().shape == (48, 2) and heldout_rows().shape == (12, 2)
assert len(trace) == 800
assert last20 <= 0.8 * trace[0]
assert len(changed) >= 1
assert guard_stats == {"calls": 800, "rows": 800 * 48}
assert all(latent_data.row_sha256(row) in TRAIN_HASHES for row in T)
assert not any(latent_data.row_sha256(row) in HELDOUT_HASHES for row in T)
# Sampling probes
assert samples.shape == (512, 2) and samples.dtype == torch.float32
assert bool(torch.isfinite(samples).all()) and samples.requires_grad is False
assert bool(((samples.mean(dim=0) - T.mean(dim=0)).abs() <= 0.5).all())
assert bool((samples.std(dim=0) >= 0.5 * T.std(dim=0)).all())
assert bool((counts >= 51).all())
# Interpolation probes
assert z_path.shape == (9, 2) and decoded_path.shape == (9, 2)
assert torch.allclose(z_path[0], mu_a[0], atol=ATOL, rtol=RTOL)
assert torch.allclose(z_path[-1], mu_b[0], atol=ATOL, rtol=RTOL)
assert all(torch.allclose(steps_diff[i], steps_diff[0], atol=ATOL, rtol=RTOL) for i in range(8))
assert torch.allclose(decoded_path[0], decoded_a[0], atol=ATOL, rtol=RTOL)
assert torch.allclose(decoded_path[-1], decoded_b[0], atol=ATOL, rtol=RTOL)